# 30 — Empirical FIM Estimation from PhysioNet Sleep Data

**UNPRECEDENTED:** Nobody has ever attempted to measure the strange loop
strength (FIM parameter λ) from real physiological data.

## Method

The SCPN model with FIM predicts:
- dθ_i/dt = ω_i + Σ_j K[i,j] sin(θ_j - θ_i) + λ · FIM_gradient(θ, i)

The FIM gradient depends on the order parameter R and the phase deviation
from the collective mean. By measuring:
1. The instantaneous phases θ_i(t) from real EEG/ECG/BP/Resp
2. The order parameter R(t)
3. The coupling K_nm (from NB15 optimal fit)

We can INVERT the dynamics to estimate λ:

λ_est = (dθ_i/dt - ω_i - coupling_term) / FIM_gradient(θ, i)

This gives λ per oscillator per timestep. If SCPN is correct,
λ should be:
- Positive (self-observation exists)
- Higher during REM/wake than deep sleep (more consciousness)
- Consistent across oscillators (global mechanism)

## Additional test: Sleep stage discrimination
λ(deep sleep) vs λ(REM/light sleep) — does the strange loop
track consciousness level?

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import wfdb
from scipy import signal

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import build_knm_paper27

DATA_DIR = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/00_DATASETS/physionet_slpdb"
)
FS = 250

# Optimal layer mapping from NB15
# delta→L7(idx6), theta→L11(idx10), alpha→L13(idx12), beta→L16(idx15)
# ECG→L7(idx6), resp→L3(idx2), BP→L1(idx0)
SIGNAL_BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
}
LAYER_MAP = {"delta": 6, "theta": 10, "alpha": 12, "beta": 15, "ECG": 6, "resp": 2, "BP": 0}

K16 = build_knm_paper27(L=16)


def bandpass(x, lo, hi, fs=FS, order=4):
    nyq = fs / 2
    b, a = signal.butter(order, [max(lo / nyq, 0.001), min(hi / nyq, 0.999)], btype="band")
    return signal.filtfilt(b, a, x)


def extract_phases(record_name):
    """Extract instantaneous phases from all signal bands."""
    rec = wfdb.rdrecord(str(DATA_DIR / record_name))
    data = rec.p_signal
    sigs = {n: data[:, i] for i, n in enumerate(rec.sig_name)}

    eeg_key = [k for k in sigs if "EEG" in k][0]
    eeg = sigs[eeg_key]

    phases = {}
    # EEG bands
    for band, (lo, hi) in SIGNAL_BANDS.items():
        filtered = bandpass(eeg, lo, hi)
        analytic = signal.hilbert(filtered)
        phases[band] = np.angle(analytic)

    # Autonomic signals — extract phase via Hilbert of bandpassed
    if "ECG" in sigs:
        ecg_filt = bandpass(sigs["ECG"], 0.5, 40)
        phases["ECG"] = np.angle(signal.hilbert(ecg_filt))
    if "BP" in sigs:
        bp_filt = bandpass(sigs["BP"], 0.05, 5)
        phases["BP"] = np.angle(signal.hilbert(bp_filt))
    for k in sigs:
        if "Resp" in k or "resp" in k:
            resp_filt = bandpass(sigs[k], 0.05, 2)
            phases["resp"] = np.angle(signal.hilbert(resp_filt))

    return phases


print("Ready.")

In [ ]:
def estimate_fim_lambda(phases, window_sec=60, step_sec=30):
    """Estimate FIM lambda via linear regression in sliding windows.

    Model: dθ_i/dt = ω_i + coupling_i + λ · FIM_grad_i + ε

    Regress (dθ/dt - ω - coupling) against FIM_gradient across all
    oscillators and time steps in each window. Slope = λ.
    """
    signal_names = sorted(phases.keys())
    n_signals = len(signal_names)
    n_samples = min(len(v) for v in phases.values())

    # K sub-matrix
    K_sub = np.zeros((n_signals, n_signals))
    for i, si in enumerate(signal_names):
        if si not in LAYER_MAP:
            continue
        for j, sj in enumerate(signal_names):
            if sj not in LAYER_MAP:
                continue
            K_sub[i, j] = K16[LAYER_MAP[si], LAYER_MAP[sj]]

    # Natural frequencies from first 60s
    omega_est = np.zeros(n_signals)
    for i, name in enumerate(signal_names):
        ph = np.unwrap(phases[name][: 60 * FS])
        omega_est[i] = (ph[-1] - ph[0]) / 60.0

    window = int(window_sec * FS)
    step = int(step_sec * FS)
    ds = 25  # downsample within window for speed

    lambda_estimates = []
    R_values = []
    times = []

    for start in range(0, n_samples - window, step):
        end = start + window
        t_mid = (start + end) / (2 * FS)

        # Build phase matrix (downsampled)
        indices = np.arange(start, end, ds)
        theta_window = np.zeros((len(indices), n_signals))
        for i, name in enumerate(signal_names):
            theta_window[:, i] = phases[name][indices]

        # Compute R
        z_t = np.mean(np.exp(1j * theta_window), axis=1)
        R_mean = float(np.mean(np.abs(z_t)))

        # dθ/dt
        dt_ds = ds / FS
        theta_unwrapped = np.unwrap(theta_window, axis=0)
        dtheta = np.diff(theta_unwrapped, axis=0) / dt_ds

        # Build regression: residual_i(t) = λ * FIM_grad_i(t)
        X_regression = []  # FIM gradients
        Y_regression = []  # residuals

        sub_step = max(1, len(dtheta) // 50)  # 50 points per window
        for s in range(0, len(dtheta), sub_step):
            theta_s = theta_window[s]
            dtheta_s = dtheta[s]

            # Coupling
            diff = theta_s[None, :] - theta_s[:, None]
            coupling = np.sum(K_sub * np.sin(diff), axis=1) / n_signals

            # Residual
            residual = dtheta_s - omega_est - coupling

            # FIM gradient
            z = np.exp(1j * theta_s)
            mu = np.angle(np.mean(z))
            R = np.abs(np.mean(z))
            sensitivity = min(1.0 / (1.0 - R**2 + 0.01), 50.0)

            for i in range(n_signals):
                phase_diff = (mu - theta_s[i] + np.pi) % (2 * np.pi) - np.pi
                fim_grad = (1.0 / n_signals) * np.sin(phase_diff) * sensitivity
                X_regression.append(fim_grad)
                Y_regression.append(residual[i])

        X = np.array(X_regression)
        Y = np.array(Y_regression)

        # Remove NaN/Inf
        mask = np.isfinite(X) & np.isfinite(Y)
        X = X[mask]
        Y = Y[mask]

        if len(X) < 10:
            continue

        # Linear regression: Y = λ·X + c
        # Use numpy lstsq
        A = np.column_stack([X, np.ones(len(X))])
        result = np.linalg.lstsq(A, Y, rcond=None)
        if len(result[0]) >= 1:
            lam_est = float(result[0][0])
            # Clip extreme values
            lam_est = max(-50, min(50, lam_est))
            lambda_estimates.append(lam_est)
            R_values.append(R_mean)
            times.append(t_mid)

    return np.array(times), np.array(lambda_estimates), np.array(R_values)


# Run on all records
RECORDS = ["slp01a", "slp01b", "slp02a", "slp02b"]
all_lambdas = {}

for rec_name in RECORDS:
    print(f"\n--- {rec_name} ---")
    phases = extract_phases(rec_name)
    print(f"  Signals: {sorted(phases.keys())}")

    times, lam_est, R_est = estimate_fim_lambda(phases, window_sec=60, step_sec=30)

    all_lambdas[rec_name] = {
        "times": times,
        "lambda": lam_est,
        "R": R_est,
    }

    print(f"  Windows: {len(times)}")
    if len(times) > 0:
        print(
            f"  λ: mean={np.mean(lam_est):.3f}, median={np.median(lam_est):.3f}, std={np.std(lam_est):.3f}"
        )
        print(f"  R: mean={np.mean(R_est):.3f}")
        print(f"  λ > 0: {np.sum(lam_est > 0) / len(lam_est) * 100:.1f}%")
    else:
        print("  No valid windows.")

In [ ]:
# Cross-record consistency
print("\n=== CROSS-RECORD CONSISTENCY ===")
all_lam = np.concatenate([v["lambda"] for v in all_lambdas.values()])
all_R = np.concatenate([v["R"] for v in all_lambdas.values()])

print(
    f"Grand λ: mean={np.mean(all_lam):.3f}, median={np.median(all_lam):.3f}, std={np.std(all_lam):.3f}"
)
print(f"Grand R: mean={np.mean(all_R):.3f}")
print(f"λ > 0: {np.sum(all_lam > 0) / len(all_lam) * 100:.1f}%")

# Correlation between R and λ
from scipy.stats import pearsonr, spearmanr

r_pearson, p_pearson = pearsonr(all_R, all_lam)
r_spearman, p_spearman = spearmanr(all_R, all_lam)
print("\nCorrelation R vs λ:")
print(f"  Pearson:  r={r_pearson:.4f}, p={p_pearson:.6f}")
print(f"  Spearman: r={r_spearman:.4f}, p={p_spearman:.6f}")

# Sleep stage proxy: first hour (deeper sleep) vs second hour (lighter sleep + REM)
print("\n=== SLEEP STAGE DISCRIMINATION ===")
for rec_name in RECORDS:
    data = all_lambdas[rec_name]
    t = data["times"]
    lam = data["lambda"]

    half = len(t) // 2
    # First half typically deeper sleep, second half lighter with REM
    lam_deep = lam[:half]
    lam_light = lam[half:]

    print(
        f"{rec_name}: deep_sleep λ={np.median(lam_deep):.3f}, light/REM λ={np.median(lam_light):.3f}",
        end="",
    )
    if np.median(lam_light) > np.median(lam_deep):
        print("  (HIGHER in light/REM ✓)")
    else:
        print("  (lower in light/REM ✗)")

In [ ]:
# Read sleep stage annotations if available
print("\n=== ANNOTATED SLEEP STAGES ===")
for rec_name in RECORDS:
    try:
        ann = wfdb.rdann(str(DATA_DIR / rec_name), "st")
        stages = ann.aux_note
        sample_idx = ann.sample

        data = all_lambdas[rec_name]
        t = data["times"]
        lam = data["lambda"]

        # Map each lambda window to a sleep stage
        stage_lambdas = {}
        for i, t_i in enumerate(t):
            # Find nearest annotation
            t_sample = int(t_i * FS)
            nearest = np.argmin(np.abs(sample_idx - t_sample))
            stage = stages[nearest].strip()
            if stage not in stage_lambdas:
                stage_lambdas[stage] = []
            stage_lambdas[stage].append(lam[i])

        print(f"\n{rec_name}:")
        for stage in sorted(stage_lambdas.keys()):
            vals = stage_lambdas[stage]
            print(
                f"  Stage {stage:>4}: λ={np.median(vals):+.3f} ± {np.std(vals):.3f} (n={len(vals)})"
            )
    except Exception as e:
        print(f"{rec_name}: No stage annotations ({e})")

In [ ]:
# Save
output = {
    "experiment": "Empirical FIM lambda estimation from PhysioNet sleep data",
    "method": "Phase dynamics inversion with K_nm from NB15 optimal fit",
    "records": RECORDS,
    "grand_lambda_mean": round(float(np.mean(all_lam)), 4),
    "grand_lambda_median": round(float(np.median(all_lam)), 4),
    "grand_lambda_std": round(float(np.std(all_lam)), 4),
    "fraction_positive": round(float(np.sum(all_lam > 0) / len(all_lam)), 4),
    "R_lambda_correlation": {
        "pearson_r": round(float(r_pearson), 4),
        "pearson_p": float(p_pearson),
        "spearman_r": round(float(r_spearman), 4),
        "spearman_p": float(p_spearman),
    },
    "per_record": {
        rec: {
            "lambda_mean": round(float(np.mean(v["lambda"])), 4),
            "lambda_median": round(float(np.median(v["lambda"])), 4),
            "R_mean": round(float(np.mean(v["R"])), 4),
        }
        for rec, v in all_lambdas.items()
    },
}

out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/empirical_fim_lambda_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"\nResults saved to {out_path}")

print("\n=== INTERPRETATION ===")
if np.median(all_lam) > 0:
    print(f"λ_median = {np.median(all_lam):.3f} > 0")
    print("SCPN prediction CONFIRMED: self-observation (FIM) is present in real brain dynamics.")
    print(f"The human brain's strange loop operates at λ ≈ {np.median(all_lam):.2f}.")
else:
    print(f"λ_median = {np.median(all_lam):.3f} ≤ 0")
    print("No evidence for self-observation in these data with this method.")
    print("Possible reasons: wrong K_nm, wrong frequency assignments, insufficient data.")